# Double Descent study — `credit` dataset

**What is double descent?** Normally we expect: train longer → training loss keeps dropping, but test loss goes down then **up** (classic overfitting), so you stop early. *Deep double descent* (Nakkiran et al., 2019) says that if you keep training **far past** the point where training loss hits ~zero, test loss can come **down a second time** — sometimes below its first minimum.

```
test loss:   \____/‾‾‾\____      ← down, UP (the bump), down again
train loss:  \_____________      ← hits ~0 somewhere in the middle
```

**The recipe used here (your settings):**

| setting | value | why |
|---|---|---|
| optimizer | **vanilla SGD** (momentum 0) | slow, plain updates — the classic setting for the phenomenon |
| learning rate | **1e-5** | very small steps |
| batch size | **16** | small batches = many noisy updates |
| epochs | **4000** | must train far past interpolation to see the second descent |
| dropout / weight-decay / L1 | **0** | regularization *suppresses* double descent, so it's off |

**Three models, all measured per epoch:** `x` (raw only), `x+tree` (raw + RF split bits), `x+tree+deep` (FULL). For every epoch we record **train / validation / test** × **loss, AUC, accuracy**.

> ⚠️ **Runtime: roughly 1 hour per model, ~3 hours total** (700 steps/epoch × 4000 epochs = 2.8M steps each). Each model runs in its **own cell** and saves its CSV when done, so a disconnect never costs you the finished ones. The script also flushes the CSV every 100 epochs. Keep the tab awake.

> Note: test metrics are recorded **only to plot the curve** — model selection never uses them.

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> Runtime > Change runtime type > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 3b · OPTIONAL fallback — only if OpenML gives 504 errors.
#      Upload openml_cache_clean5.tar.gz from your Desktop; otherwise press Cancel.
import os, glob, tarfile
try:
    from google.colab import files
    files.upload()
except Exception as e:
    print('skipped:', e)
hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
if hits:
    dst = '/root/.cache/openml/org/openml/www'; os.makedirs(dst, exist_ok=True)
    with tarfile.open(hits[0]) as t: t.extractall(dst)
    print('cache extracted — running fully offline')
else:
    print('no bundle — will download from OpenML (fine if it is up)')

In [ ]:
# 4 · the shared configuration (your exact settings)
CFG = ('--task 361055 '            # credit
       '--optimizer sgd --momentum 0 '   # vanilla SGD
       '--lr 1e-5 '
       '--batch-size 16 '
       '--epochs 4000 '
       '--dropout 0 --weight-decay 0 --l1 0 '   # no regularization
       '--eval-every 1 '                        # metrics EVERY epoch
       '--log-every 100 --flush-every 100 '
       '--device auto')
print(CFG)

## Run the three models
Each cell takes roughly an hour. They save independently — if one dies, the others' results survive.

In [ ]:
# 5 · MODEL 1 of 3 — raw features only (the baseline)
!python -u run_double_descent.py {CFG} --views 'x'

In [ ]:
# 6 · MODEL 2 of 3 — raw + tree split-bits
!python -u run_double_descent.py {CFG} --views 'x+tree'

In [ ]:
# 7 · MODEL 3 of 3 — FULL (raw + tree + deep extractor)
!python -u run_double_descent.py {CFG} --views 'x+tree+deep'

## Combine and analyse

In [ ]:
# 8 · merge the three histories + build the comparison figure
import os, json, pandas as pd
from run_double_descent import compare_figure
D = 'results/double_descent'
paths = {'x': f'{D}/dd_credit_x.csv',
         'x+tree': f'{D}/dd_credit_x-tree.csv',
         'x+tree+deep': f'{D}/dd_credit_x-tree-deep.csv'}
have = {k: v for k, v in paths.items() if os.path.exists(v)}
print('found:', list(have))
df = pd.concat([pd.read_csv(v) for v in have.values()], ignore_index=True)
df.to_csv(f'{D}/dd_credit_epochs.csv', index=False)
ceiling = json.load(open(f'{D}/dd_credit.json'))['tree_ceiling']
compare_figure(df, ceiling, f'{D}/dd_credit_compare.png')
print('rows:', len(df), '| tree ceiling:', round(ceiling, 4))

In [ ]:
# 9 · DOUBLE-DESCENT LANDMARKS  (does the test loss dip -> rise -> dip again?)
for m in df.model.unique():
    g = df[df.model == m].sort_values('epoch').reset_index(drop=True)
    i1 = g.test_loss.idxmin()
    first_half = g[g.epoch <= g.epoch.max() / 2]
    iA = first_half.test_loss.idxmin()                 # first dip
    after = g.loc[iA:]
    iP = after.test_loss.idxmax()                      # the bump peak
    tail = g.loc[iP:]
    iB = tail.test_loss.idxmin()                       # second dip
    print(f'\n{m}')
    print(f"   first dip : ep {int(g.loc[iA,'epoch']):5d}  test_loss={g.loc[iA,'test_loss']:.4f}  auc={g.loc[iA,'test_auc']:.4f}")
    print(f"   bump peak : ep {int(g.loc[iP,'epoch']):5d}  test_loss={g.loc[iP,'test_loss']:.4f}")
    print(f"   later dip : ep {int(g.loc[iB,'epoch']):5d}  test_loss={g.loc[iB,'test_loss']:.4f}  auc={g.loc[iB,'test_auc']:.4f}")
    rise = g.loc[iP,'test_loss'] - g.loc[iA,'test_loss']
    fall = g.loc[iP,'test_loss'] - g.loc[iB,'test_loss']
    print(f'   rise after first dip = {rise:+.4f} | fall after peak = {fall:+.4f}')
    print(f"   final train_loss = {g.test_loss.iloc[-1]:.4f} (test), {g.train_loss.iloc[-1]:.5f} (train)")
    verdict = ('LOOKS LIKE DOUBLE DESCENT' if rise > 0.005 and fall > 0.005
               else 'no clear second descent (single descent / plain overfitting)')
    print('   ->', verdict)

In [ ]:
# 10 · show every figure
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/double_descent/*.png')):
    print('==', f, '==')
    display(Image(f))

In [ ]:
# 11 · download everything
import shutil
from google.colab import files
shutil.make_archive('double_descent_credit', 'zip', 'results/double_descent')
files.download('double_descent_credit.zip')